# Actividad Spark: MapReduce y Exploración Distribuida sobre Reseñas

Fecha: 17 de abril de 2026

## Objetivos de la práctica
1. Cargar un conjunto de datos grande en formato CSV con Spark.
2. Aplicar MapReduce (RDD) para conteo de palabras en reviews.text.
3. Evaluar particionamiento y balanceo de carga en Spark.
4. Comparar tiempos de ejecución con diferentes números de particiones.
5. Realizar consultas distribuidas con la API de DataFrames.

## Nota de alcance
Para esta práctica se usa 7817_1.csv, de acuerdo con la aclaración del profesor.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import re
import os
import time
import statistics

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("ActividadSpark_MapReduce_DataFrames")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
sc = spark.sparkContext

print("Spark inicializado correctamente")
print(f"Spark version: {spark.version}")
print(f"Default parallelism: {sc.defaultParallelism}")

26/04/17 23:20:33 WARN Utils: Your hostname, jhonh-HP-Pavilion-Gaming-Laptop-15-dk1xxx resolves to a loopback address: 127.0.1.1; using 192.168.1.12 instead (on interface wlo1)
26/04/17 23:20:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/17 23:20:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark inicializado correctamente
Spark version: 3.5.2
Default parallelism: 8


In [2]:
import sys
import pyspark

print("Kernel Python:", sys.executable)
print("PySpark version:", pyspark.__version__)
print("Si este path apunta a .venv/bin/python, estás usando el entorno virtual correcto.")

Kernel Python: /home/jhonh/Code/iue/disease-detector-notifier/.venv/bin/python
PySpark version: 3.5.2
Si este path apunta a .venv/bin/python, estás usando el entorno virtual correcto.


## 1) Carga del dataset CSV y preparación de texto

Se carga el archivo `7817_1.csv` como DataFrame y se normaliza la columna `reviews.text` para trabajar únicamente con reseñas válidas.

In [3]:
path = "7817_1.csv"
if not os.path.exists(path):
    raise FileNotFoundError(f"No se encontró el archivo en la ruta: {path}")

df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(path)
)

# Estas columnas tienen punto en el nombre literal, por eso se escapan con backticks.
reviews_df = (
    df.select(
        F.col("`reviews.text`").alias("review_text"),
        F.col("`reviews.rating`").cast("double").alias("rating"),
        F.col("`reviews.doRecommend`").alias("do_recommend"),
        F.col("brand"),
        F.col("dateAdded")
    )
    .filter(F.col("review_text").isNotNull())
    .filter(F.length(F.trim(F.col("review_text"))) > 0)
)

reviews_df.cache()
total_reviews = reviews_df.count()
print(f"Total de reseñas válidas: {total_reviews}")
reviews_df.show(5, truncate=100)

Total de reseñas válidas: 1597
+--------------------+------+--------------------+------+-------------------+
|         review_text|rating|        do_recommend| brand|          dateAdded|
+--------------------+------+--------------------+------+-------------------+
|""isSale"":""false""|  NULL|""currency"":""USD""|Amazon|2016-03-08 15:21:53|
|""isSale"":""false""|  NULL|""currency"":""USD""|Amazon|2016-03-08 15:21:53|
|""isSale"":""false""|  NULL|""currency"":""USD""|Amazon|2016-03-08 15:21:53|
|""isSale"":""false""|  NULL|""currency"":""USD""|Amazon|2016-03-08 15:21:53|
|""isSale"":""false""|  NULL|""currency"":""USD""|Amazon|2016-03-08 15:21:53|
+--------------------+------+--------------------+------+-------------------+
only showing top 5 rows



## 2) MapReduce (RDD) para conteo de palabras en `reviews.text`

Se transforma la columna de reseñas a RDD y se aplica el flujo clásico `map -> shuffle/reduce` con `flatMap` y `reduceByKey`.

In [4]:
stopwords = {
    "the", "and", "to", "of", "a", "in", "is", "it", "for", "this", "that", "i", "my", "was", "on",
    "with", "as", "very", "but", "have", "are", "so", "an", "be", "if", "its", "at", "from"
}

def tokenize(text):
    clean_text = re.sub(r"[^a-zA-Z\s]", " ", text.lower())
    return [w for w in clean_text.split() if len(w) > 2 and w not in stopwords]

reviews_rdd = reviews_df.select("review_text").rdd.map(lambda row: row[0])
words_rdd = reviews_rdd.flatMap(tokenize)
word_counts_rdd = words_rdd.map(lambda w: (w, 1)).reduceByKey(lambda x, y: x + y)

top_words = word_counts_rdd.takeOrdered(20, key=lambda x: -x[1])
print("Top 20 palabras más frecuentes en reviews.text")
for word, count in top_words:
    print(f"{word}: {count}")

Top 20 palabras más frecuentes en reviews.text
false: 252
issale: 252
shipping: 126
amazon: 119
com: 115
free: 63
merchant: 60
sourceurls: 55
www: 55
https: 36
ref: 36
kink: 32
avail: 32
product: 32
con: 32
kindle: 20
http: 19
paperwhite: 16
display: 15
resolution: 12


## 3) Particiones, balanceo de carga y comparación de tiempos

Se prueba el mismo flujo de conteo con distintos números de particiones.
Además del tiempo total, se mide la distribución de carga por partición para detectar desbalance (skew).

In [5]:
def partition_sizes(rdd):
    return rdd.mapPartitionsWithIndex(
        lambda idx, it: [(idx, sum(1 for _ in it))]
    ).collect()

def run_wordcount_with_partitions(base_reviews_rdd, num_parts):
    working_rdd = base_reviews_rdd.repartition(num_parts)
    parts_info = partition_sizes(working_rdd)

    start = time.perf_counter()
    result_count = (
        working_rdd
        .flatMap(tokenize)
        .map(lambda w: (w, 1))
        .reduceByKey(lambda x, y: x + y)
        .count()
    )
    duration = time.perf_counter() - start

    sizes_only = [size for _, size in parts_info]
    return {
        "partitions": num_parts,
        "duration_sec": duration,
        "distinct_words": result_count,
        "min_partition_size": min(sizes_only) if sizes_only else 0,
        "max_partition_size": max(sizes_only) if sizes_only else 0,
        "avg_partition_size": (sum(sizes_only) / len(sizes_only)) if sizes_only else 0.0,
        "std_partition_size": statistics.pstdev(sizes_only) if len(sizes_only) > 1 else 0.0,
        "partition_sizes": parts_info,
    }

candidate_partitions = [2, 4, 8, 16, sc.defaultParallelism]
candidate_partitions = sorted(set([p for p in candidate_partitions if p > 0]))

benchmark_rows = []
repetitions = 3

for p in candidate_partitions:
    runs = [run_wordcount_with_partitions(reviews_rdd, p) for _ in range(repetitions)]
    mean_time = sum(r["duration_sec"] for r in runs) / repetitions
    benchmark_rows.append({
        "partitions": p,
        "mean_duration_sec": mean_time,
        "best_duration_sec": min(r["duration_sec"] for r in runs),
        "worst_duration_sec": max(r["duration_sec"] for r in runs),
        "std_partition_size": runs[0]["std_partition_size"],
        "min_partition_size": runs[0]["min_partition_size"],
        "max_partition_size": runs[0]["max_partition_size"],
        "distinct_words": runs[0]["distinct_words"],
    })

benchmark_df = spark.createDataFrame(benchmark_rows)
benchmark_df.orderBy("mean_duration_sec").show(truncate=False)

+-------------------+--------------+------------------+-------------------+------------------+----------+------------------+-------------------+
|best_duration_sec  |distinct_words|max_partition_size|mean_duration_sec  |min_partition_size|partitions|std_partition_size|worst_duration_sec |
+-------------------+--------------+------------------+-------------------+------------------+----------+------------------+-------------------+
|0.2301181099974201 |34            |799               |0.2803080206649611 |798               |2         |0.5               |0.312971851999464  |
|0.31489488700026413|34            |414               |0.35980060766693595|386               |4         |9.98436277385793  |0.38448932800019975|
|0.4273473989997001 |34            |210               |0.4701652670009935 |190               |8         |6.613197033205649 |0.5190922420006245 |
|0.7853523039993888 |34            |110               |0.8422728033328895 |90                |16        |6.074729932268594 |0.9431

In [6]:
best_row = benchmark_df.orderBy("mean_duration_sec").first()
print(f"Mejor configuración observada: {best_row['partitions']} particiones")
print(f"Tiempo promedio: {best_row['mean_duration_sec']:.4f} s")

print("\nDetalle de distribución para la mejor configuración (index, tamaño):")
best_dist = run_wordcount_with_partitions(reviews_rdd, int(best_row["partitions"]))
for idx, size in best_dist["partition_sizes"]:
    print(f"Partición {idx}: {size}")

Mejor configuración observada: 2 particiones
Tiempo promedio: 0.2803 s

Detalle de distribución para la mejor configuración (index, tamaño):
Partición 0: 798
Partición 1: 799


## 4) Consultas distribuidas con DataFrames

Las siguientes consultas usan la API de DataFrames para explorar el dataset de reseñas en paralelo.
Se incluyen agregaciones por marca, rating y recomendación.

In [7]:
# 4.1 Distribución de ratings
reviews_df.groupBy("rating").count().orderBy(F.col("count").desc()).show(20, truncate=False)

# 4.2 Porcentaje de reseñas recomendadas/no recomendadas
recommendation_stats = (
    reviews_df
    .groupBy("do_recommend")
    .count()
    .withColumn("pct", F.round(F.col("count") * 100.0 / total_reviews, 2))
    .orderBy(F.col("count").desc())
)
recommendation_stats.show(truncate=False)

# 4.3 Top marcas por número de reseñas
reviews_df.groupBy("brand").count().orderBy(F.col("count").desc()).show(15, truncate=False)

# 4.4 Rating promedio por marca (mínimo 50 reseñas para evitar ruido)
(
    reviews_df
    .groupBy("brand")
    .agg(
        F.count("*").alias("n_reviews"),
        F.round(F.avg("rating"), 3).alias("avg_rating")
    )
    .filter(F.col("n_reviews") >= 50)
    .orderBy(F.col("avg_rating").desc(), F.col("n_reviews").desc())
    .show(15, truncate=False)
)

# 4.5 Longitud promedio de reseñas
reviews_df.select(F.avg(F.length("review_text")).alias("avg_review_length_chars")).show()

print("Plan físico de una consulta de agregación (evidencia de ejecución distribuida):")
reviews_df.groupBy("brand").count().explain(mode="formatted")

+------+-----+
|rating|count|
+------+-----+
|NULL  |1597 |
+------+-----+

+---------------------+-----+----+
|do_recommend         |count|pct |
+---------------------+-----+----+
|""currency"":""USD"" |1260 |78.9|
|""condition"":""New""|337  |21.1|
+---------------------+-----+----+

+------+-----+
|brand |count|
+------+-----+
|Amazon|1585 |
|Moshi |12   |
+------+-----+

+------+---------+----------+
|brand |n_reviews|avg_rating|
+------+---------+----------+
|Amazon|1585     |NULL      |
+------+---------+----------+

+-----------------------+
|avg_review_length_chars|
+-----------------------+
|      29.52222917971196|
+-----------------------+

Plan físico de una consulta de agregación (evidencia de ejecución distribuida):
== Physical Plan ==
AdaptiveSparkPlan (9)
+- HashAggregate (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- InMemoryTableScan (1)
               +- InMemoryRelation (2)
                     +- * Project (5)
                        +- * Filter (4)


## 5) Cierre técnico

Este notebook cumple los puntos de la actividad sobre CSV, MapReduce, particionamiento y DataFrames.
Recomendación para el informe: incluir capturas de salida de cada sección y comentar la configuración de particiones que obtuvo mejor tiempo en tu equipo.